# IFB-451 Big Data: Modul 4 — Hadoop Ecosystem
## Proyek: Sistem Analisis Prediktif dan Pemetaan Sebaran Risiko Stunting
**Dosen Pengampu:** Dr. Edi Nuryatno  
**Fokus Studi:** Health Informatics (Data Stunting Balita Indonesia)

---

### Target Pembelajaran (Learning Outcomes Modul 4):
1. Memahami arsitektur dan komponen utama ekosistem **Hadoop** (HDFS, YARN, MapReduce, Hive, Pig, HBase, Sqoop, Flume, Oozie).
2. Memahami pembagian peran: *Storage* (HDFS), *Resource Management* (YARN), dan *Distributed Processing* (MapReduce & Hive).
3. Menjalankan *job* pemrosesan data terdistribusi menggunakan **MapReduce Python Streaming** (`mapper.py` dan `reducer.py`).
4. Mengimplementasikan *data warehousing* dan kueri analitik menggunakan **Apache Hive (HiveQL)**.
5. Menganalisis studi kasus kesehatan nasional (*National Health & Stunting Registry*) dan integrasi arsitektur Big Data.

## 1. Landasan Teori: Ekosistem Apache Hadoop

Ekosistem Hadoop dirancang untuk memecahkan permasalahan pemrosesan data berskala besar (*Big Data*) menggunakan klaster komputer komoditas (*commodity hardware*).

### Komponen Utama Ekosistem:
1. **HDFS (Hadoop Distributed File System):** Lapisan penyimpanan terdistribusi yang membagi file besar menjadi blok-blok (standar 128 MB) dan menduplikasikannya secara otomatis (*default replication factor 3x*) untuk menjamin *fault tolerance*.
2. **YARN (Yet Another Resource Negotiator):** Mengelola alokasi CPU dan memori RAM pada klaster serta menjadwalkan pekerjaan (*job scheduling*).
3. **MapReduce:** Model pemrograman pemrosesan paralel berbasis *batch* dengan fase *Map* (pemetaan/filtering) dan *Reduce* (agregasi).
4. **Apache Hive:** Data warehouse berbasis SQL di atas Hadoop; memungkinkan pengguna mengeksekusi kueri terstruktur (*HiveQL*) yang secara otomatis diterjemahkan menjadi pekerjaan MapReduce/Tez.
5. **Apache Pig:** Platform pemrosesan data berbasis aliran data (*dataflow language*) menggunakan bahasa *Pig Latin* untuk operasi ETL.
6. **Apache HBase:** Database NoSQL berorientasi kolom (*column-oriented*) untuk operasi baca/tulis cepat (*low latency*) pada data berskala miliaran baris.
7. **Apache Sqoop & Flume:** Tool ingestion data. Sqoop mentransfer data dari RDBMS (MySQL, PostgreSQL) ke HDFS, sedangkan Flume menyerap data streaming/log secara terus-menerus.
8. **Apache Oozie:** Penjadwal dan orkestrasi alur kerja (*workflow scheduler*) antar komponen Hadoop.

## 2. Ingestion & Pembersihan Data Stunting Indonesia
Dataset yang digunakan adalah data prevalensi balita stunting di 38 Provinsi Indonesia (periode 2020 - 2023). File mentah CSV memiliki format satu baris dengan pemisah titik koma (`;`) dan desimal koma (`,`). Pada tahap ini, kita memuat dan membersihkan data menjadi format terstruktur standar.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# 1. Membaca dataset mentah
raw_csv_path = 'data_stunting_clean.csv'
if not os.path.exists(raw_csv_path):
    # Jika dijalankan dari root atau subfolder
    raw_csv_path = 'data/data_stunting_clean.csv'

df = pd.read_csv(raw_csv_path)

print("=== INFORMASI DATASET PREVALENSI STUNTING INDONESIA ===")
print(f"Total Wilayah (Provinsi): {len(df)}")
print(f"Rentang Periode Data: 2020 s.d. 2023")
print("\n5 Baris Teratas Dataset:")
display(df.head())

print("\nRingkasan Statistik Prevalensi Stunting (%):")
display(df.describe())

## 3. HDFS (Hadoop Distributed File System)
HDFS mengelola penyimpanan data secara terdistribusi. Berikut adalah simulasi dan perintah bash resmi yang digunakan untuk mengunggah dataset stunting ke dalam HDFS.

In [ ]:
# Menampilkan Perintah CLI HDFS yang digunakan dalam Lingkungan Produksi/Klaster Hadoop
hdfs_commands = """
# 1. Membuat direktori tujuan pada HDFS
hdfs dfs -mkdir -p /user/hadoop/stunting/input

# 2. Mengunggah file dataset CSV dari filesystem lokal ke HDFS
hdfs dfs -put data_stunting_clean.csv /user/hadoop/stunting/input/

# 3. Memverifikasi isi file dan ukuran di dalam HDFS
hdfs dfs -ls /user/hadoop/stunting/input/

# 4. Melihat 10 baris pertama data di HDFS
hdfs dfs -cat /user/hadoop/stunting/input/data_stunting_clean.csv | head -n 10
"""
print("=== DAFTAR PERINTAH HADOOP HDFS (CLI) ===")
print(hdfs_commands)

## 4. Pemrosesan Terdistribusi dengan MapReduce (Hadoop Streaming Python)
Sesuai panduan Modul 4 halaman 34, pemrosesan batch MapReduce dapat diimplementasikan dalam bahasa Python menggunakan **Hadoop Streaming**.

- **`mapper.py`**: Membaca setiap baris data provinsi, mengambil angka prevalensi tahun 2023, dan memetakannya ke dalam pasangan key-value: `<Kategori_Risiko>\t1`.
- **`reducer.py`**: Menjumlahkan total provinsi di masing-masing kategori risiko stunting.

In [ ]:
# Simulasi Eksekusi MapReduce secara Internal di Lingkungan Python
from collections import defaultdict

# 1. Fase MAPPER: Memetakan baris menjadi key-value (<Kategori>\t1)
map_outputs = []
for index, row in df.iterrows():
    val_2023 = float(row['2023'])
    if val_2023 >= 15.0:
        kategori = "RISIKO_TINGGI (>= 15%)"
    elif val_2023 >= 5.0:
        kategori = "RISIKO_SEDANG (5.0% - 14.9%)"
    else:
        kategori = "RISIKO_RENDAH (< 5.0%)"
    map_outputs.append((kategori, 1))

print(f"Contoh Keluaran Mapper (5 record pertama): {map_outputs[:5]}")

# 2. Fase SHUFFLE & SORT: Pengelompokan berdasarkan Key
map_outputs.sort(key=lambda x: x[0])

# 3. Fase REDUCER: Agregasi frekuensi
reduce_counts = defaultdict(int)
for key, val in map_outputs:
    reduce_counts[key] += val

print("\n" + "="*60)
print("HASIL EKSEKUSI MAPREDUCE (REDUCER OUTPUT)")
print("="*60)
for kat, jml in reduce_counts.items():
    print(f"{kat.ljust(30)} : {jml} Provinsi")
print("="*60)

### Perintah Menjalankan MapReduce di Klaster Hadoop:
```bash
hadoop jar /usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-*.jar \
    -input /user/hadoop/stunting/input/data_stunting_clean.csv \
    -output /user/hadoop/stunting/output_mapreduce \
    -mapper "python3 mapper.py" \
    -reducer "python3 reducer.py"
```

## 5. Data Warehousing & Kueri Analitik dengan Apache Hive (HiveQL)
Apache Hive menyediakan antarmuka SQL-like (*HiveQL*) di atas HDFS. Untuk mendemonstrasikan kueri Hive secara langsung di notebook, kita menggunakan engine SQL lokal yang mengeksekusi sintaksis standar yang sama persis dengan HiveQL.

In [ ]:
import sqlite3

# Membuat koneksi in-memory database untuk simulasi HiveQL
conn = sqlite3.connect(':memory:')
df.columns = ['provinsi', 'th_2020', 'th_2021', 'th_2022', 'th_2023']
df.to_sql('prevalensi_stunting', conn, index=False, if_exists='replace')

def run_hiveql(query_title, sql_statement):
    print(f"\n--- {query_title} ---")
    res = pd.read_sql_query(sql_statement, conn)
    display(res)
    return res

# Query 1: Top 5 Provinsi Tertinggi Stunting 2023 (Zona Merah Prioritas)
q1 = run_hiveql(
    "Query 1: Top 5 Provinsi Tertinggi Stunting 2023",
    """
    SELECT provinsi, th_2023 AS prevalensi_2023
    FROM prevalensi_stunting
    ORDER BY th_2023 DESC
    LIMIT 5;
    """
)

# Query 2: Top 5 Provinsi Terendah Stunting 2023 (Zona Hijau Terkendali)
q2 = run_hiveql(
    "Query 2: Top 5 Provinsi Terendah Stunting 2023",
    """
    SELECT provinsi, th_2023 AS prevalensi_2023
    FROM prevalensi_stunting
    ORDER BY th_2023 ASC
    LIMIT 5;
    """
)

# Query 3: Efektivitas Penurunan Stunting Tertinggi (2020 vs 2023)
q3 = run_hiveql(
    "Query 3: Penurunan Stunting Paling Signifikan (2020 s.d. 2023)",
    """
    SELECT provinsi, th_2020, th_2023, ROUND(th_2020 - th_2023, 2) AS penurunan_poin
    FROM prevalensi_stunting
    ORDER BY penurunan_poin DESC
    LIMIT 5;
    """
)

# Query 4: Agregasi Rata-rata Tren Nasional 2020 - 2023
q4 = run_hiveql(
    "Query 4: Rata-rata Prevalensi Stunting Nasional (2020-2023)",
    """
    SELECT 
        ROUND(AVG(th_2020), 2) AS rata_nasional_2020,
        ROUND(AVG(th_2021), 2) AS rata_nasional_2021,
        ROUND(AVG(th_2022), 2) AS rata_nasional_2022,
        ROUND(AVG(th_2023), 2) AS rata_nasional_2023
    FROM prevalensi_stunting;
    """
)

## 6. Visualisasi Hasil Analisis Data Stunting

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Grafik 1: Tren Rata-rata Nasional (2020 - 2023)
years = ['2020', '2021', '2022', '2023']
avg_values = [df['th_2020'].mean(), df['th_2021'].mean(), df['th_2022'].mean(), df['th_2023'].mean()]

axes[0].plot(years, avg_values, marker='o', linewidth=2.5, color='#2b5c8f', label='Rata-rata Nasional')
axes[0].set_title('Tren Penurunan Rata-rata Prevalensi Stunting Nasional (%)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Tahun', fontsize=11)
axes[0].set_ylabel('Prevalensi Stunting (%)', fontsize=11)
axes[0].grid(True, linestyle='--', alpha=0.6)
for x, y in zip(years, avg_values):
    axes[0].annotate(f"{y:.2f}%", (x, y), textcoords="offset points", xytext=(0, 10), ha='center', fontweight='bold')

# Grafik 2: Perbandingan Top 5 Wilayah Tertinggi vs Terendah 2023
top5 = df.sort_values(by='th_2023', ascending=False).head(5)
low5 = df.sort_values(by='th_2023', ascending=True).head(5)
combined = pd.concat([top5, low5])
colors = ['#d9534f']*5 + ['#5cb85c']*5

axes[1].barh(combined['provinsi'], combined['th_2023'], color=colors)
axes[1].set_title('Top 5 Provinsi Prevalensi Tertinggi (Merah) vs Terendah (Hijau) 2023', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Prevalensi Stunting (%)', fontsize=11)
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 7. Studi Kasus Modul 4 & Arsitektur Solusi Big Data

### Studi Kasus Modul: *National Health & Immunization Registry*
Pada modul perkuliahan, studi kasus mengkaji registri imunisasi dan kesehatan anak berskala nasional untuk melacak cakupan vaksinasi dan pola risiko.

### Integrasi Solusi dalam Proyek Sistem Stunting Kelompok:
1. **Penyimpanan Terdistribusi (HDFS):** Dataset antropometri balita jutaan baris (seperti `stunting_nasional_1gb.csv` dari Modul 1 & 2) disimpan aman di HDFS dengan replikasi 3x.
2. **Pembersihan & Transformasi (Apache Pig):** Digunakan untuk melakukan *ETL* (Extract, Transform, Load) awal, seperti memfilter data anak yang hilang (*missing values*) dan menggabungkan (*join*) data riwayat ASI eksklusif dan sanitasi.
3. **Agregasi Makro (MapReduce & Hive):** Digunakan untuk agregasi bulanan, kueri analitik kesehatan masyarakat, dan penentuan zona intervensi wilayah.
4. **Akses Data Cepat / Low Latency (Apache HBase):** Database NoSQL berorientasi kolom untuk pencarian rekam medis anak individu secara instan (*point lookup*) saat balita datang ke Posyandu.
5. **Otomasi Aliran Kerja (Apache Oozie):** Mengorkestrasikan seluruh pipeline mulai dari penyerapan data (*Sqoop/Flume*), pemrosesan Pig/MapReduce, hingga penyimpanan di Hive dan HBase secara terjadwal setiap akhir bulan.

## 8. Ringkasan Temuan & Kesimpulan
1. **Keberhasilan Penurunan Stunting Nasional:** Rata-rata prevalensi stunting nasional mengalami tren penurunan yang konsisten dari tahun 2020 (12.39%) hingga mencapai 8.16% pada tahun 2023.
2. **Wilayah Prioritas Penanganan:** Terdapat wilayah dengan prevalensi stunting yang masih tergolong tinggi di atas 15% pada tahun 2023 (seperti Sulawesi Barat 23.2%, Kalimantan Barat 17.5%, dan NTT 17.4%) yang membutuhkan intervensi terdistribusi lebih intensif.
3. **Efektivitas Ekosistem Hadoop:** Penggunaan HDFS, MapReduce, dan Apache Hive memberikan solusi komputasi yang terdistribusi, terukur (*scalable*), dan terstruktur untuk mengelola data kesehatan skala nasional tanpa terkendala keterbatasan server tunggal.